# compilar-versiculos-teste.ipynb — Corta e concatena versículos não consecutivos

Pega um capítulo já narrado (áudio + roteiro por versículo) e monta um **áudio + roteiro compilado**, só com os versículos que você escolher, na ordem que você quiser -- não precisa ser sequencial.

Gera 3 arquivos:
- **`.wav`** -- áudio compilado (só os trechos escolhidos, colados)
- **`.srt`** -- legenda recalculada do zero (tempo do compilado, não do capítulo original)
- **`.json`** -- manifesto: pra cada trecho do compilado, de qual versículo ORIGINAL ele veio -- usado depois pra montagem de vídeo (saber qual mídia buscar na biblioteca_match pra cada trecho)

**Esse notebook é um teste** -- só gera o áudio+roteiro compilados, não monta vídeo (isso é o próximo passo, depois de confirmar que essa parte funciona do jeito que você quer).

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SETUP                                                       ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
_pasta_modulos = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos"
for _arquivo in Path(_pasta_modulos).glob("*.py"):
    shutil.copy(_arquivo, ".")

!ffmpeg -version | head -1
print("✅ Setup pronto")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
✅ Setup pronto


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  CONFIGURAÇÃO                                                 ║
# ╚═══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"   # pasta em videos/ -- ex: videos/40_Matt_02/
LIVRO_PT = "Mateus"
CAPITULO = 2

# Quais versículos entram, NA ORDEM que você quer -- não precisa ser
# sequencial nem consecutivo (o teste que já validei foi exatamente
# esse: 1, 9, 23 alternados)
VERSICULOS_ALVO = [1, 9, 23]

NOME_BASE_SAIDA = "compilado_teste"   # vira compilado_teste.wav/.srt/.json

PASTA_VIDEO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}"
CAMINHO_AUDIO = f"{PASTA_VIDEO}/{NOME_ORACAO}_audio.wav"
CAMINHO_SRT_VERSICULO = f"{PASTA_VIDEO}/{NOME_ORACAO}_versiculo_multilingue.srt"
CAMINHO_ROTEIRO_TXT = f"{PASTA_VIDEO}/{NOME_ORACAO}_roteiro_versiculos.txt"

PASTA_SAIDA_LOCAL = "saida_compilacao"   # fica no Colab -- sobe pro Drive na célula final

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
print(f"   Oração:      {NOME_ORACAO}")
print(f"   Versículos:  {LIVRO_PT} {CAPITULO}:{VERSICULOS_ALVO}")
print(f"   Áudio:       {CAMINHO_AUDIO}")
print(f"   SRT versíc.: {CAMINHO_SRT_VERSICULO}")
print(f"   Roteiro:     {CAMINHO_ROTEIRO_TXT}")
print("=" * 60)

⚙️  CONFIGURAÇÃO
   Oração:      40_Matt_02
   Versículos:  Mateus 2:[1, 9, 23]
   Áudio:       /content/drive/MyDrive/narrated_video/videos/40_Matt_02/40_Matt_02_audio.wav
   SRT versíc.: /content/drive/MyDrive/narrated_video/videos/40_Matt_02/40_Matt_02_versiculo_multilingue.srt
   Roteiro:     /content/drive/MyDrive/narrated_video/videos/40_Matt_02/40_Matt_02_roteiro_versiculos.txt


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  COMPILAR                                                    ║
# ╚═══════════════════════════════════════════════════════════════════╝
from compilacao_pipeline import compilar_versiculos

for caminho in [CAMINHO_AUDIO, CAMINHO_SRT_VERSICULO, CAMINHO_ROTEIRO_TXT]:
    if not Path(caminho).exists():
        raise FileNotFoundError(f"Não achei: {caminho}")

caminho_audio_final, caminho_srt_final, caminho_manifesto_final = compilar_versiculos(
    CAMINHO_AUDIO, CAMINHO_SRT_VERSICULO, CAMINHO_ROTEIRO_TXT,
    livro_pt=LIVRO_PT, capitulo=CAPITULO, versiculos_alvo=VERSICULOS_ALVO,
    pasta_saida=PASTA_SAIDA_LOCAL, nome_base=NOME_BASE_SAIDA,
)

print(f"✅ Áudio:      {caminho_audio_final}")
print(f"✅ SRT:        {caminho_srt_final}")
print(f"✅ Manifesto:  {caminho_manifesto_final}")

✅ Áudio:      saida_compilacao/compilado_teste.wav
✅ SRT:        saida_compilacao/compilado_teste.srt
✅ Manifesto:  saida_compilacao/compilado_teste.json


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3️⃣  CONFERIR — mostra o SRT e o manifesto gerados                ║
# ╚══════════════════════════════════════════════════════════════════╝
print("="*40, "SRT COMPILADO", "="*40)
print(open(caminho_srt_final, encoding="utf-8").read())

print("="*40, "MANIFESTO", "="*40)
print(open(caminho_manifesto_final, encoding="utf-8").read())

print("="*40, "OUVIR", "="*40)
from IPython.display import Audio, display
display(Audio(caminho_audio_final))

Output hidden; open in https://colab.research.google.com to view.

In [5]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  4️⃣  SALVAR NO DRIVE (opcional -- rode se o resultado tiver bom)  ║
# ╚═════════════════════════════════════════════════════════════════╗
from drive_utils import DriveClient

_drive = DriveClient.get()
PASTA_DESTINO_DRIVE = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}/compilacoes"

for caminho in [caminho_audio_final, caminho_srt_final, caminho_manifesto_final]:
    _drive.upload(caminho, PASTA_DESTINO_DRIVE)

print(f"✅ Salvos em: {PASTA_DESTINO_DRIVE}")

✅ Salvos em: /content/drive/MyDrive/narrated_video/videos/40_Matt_02/compilacoes
